# PE6201 A2 D6 - Problem A cost-to-serve calculator

Adapted from the supplied Class 5 Capsule 2 cost-to-serve notebook. Its three-layer, sensitivity and break-even functions are retained; the Problem A cell below uses measured D5 tokens and D4 outcomes.


## Section 1 · CONFIG — every price lives here and nowhere else

Change a number in this cell and the whole model re-costs. Nothing is hardcoded further down.

Each price carries the date it was checked, because these move: **Gemini 3.7 Flash launched 13 Aug 2026
on a 50% introductory discount that expires 31 Dec 2026.** A hardcoded price is already wrong.

In [ ]:
# ── PRICES · US dollars per 1,000,000 tokens · checked 2026-08-28 against vendor pricing pages ──
PRICES = {
    #  name                    input    output   cached_input (≈0.1× input where the vendor supports it)
    "cheap":    {"in":  0.10, "out":  0.40, "cached_in": 0.01},   # e.g. Gemini 2.5 Flash-Lite tier
    "frontier": {"in":  5.00, "out": 25.00, "cached_in": 0.50},   # e.g. Claude Opus tier
}

# ── WHAT A FAILURE COSTS · US dollars per escalation ──
#    A failure is not free just because no human is named in the design.
LABOUR = {
    "support_agent": {"usd_per_hour": 45.0, "minutes_per_escalation": 8.0},
    "finance_clerk": {"usd_per_hour": 40.0, "minutes_per_escalation": 0.75},
}

def escalation_cost(role):
    "Cost of one human fallback, in US dollars."
    r = LABOUR[role]
    return r["usd_per_hour"] * r["minutes_per_escalation"] / 60.0

print("support escalation  US$%.2f" % escalation_cost("support_agent"))
print("clerk escalation    US$%.2f" % escalation_cost("finance_clerk"))
# A2 Problem A values recorded for the Mistral V2 battery.
PRICES['mistral_a2'] = {'in': 0.075, 'out': 0.20, 'cached_in': 0.075}
LABOUR['claims_assessor'] = {'usd_per_hour': 38.0, 'minutes_per_escalation': 12.0}


## Section 2 · The three-layer model

**Layer 1 · per-task variable** — fresh input + cached input + output + retrieval + tool fees.
**Layer 2 · per-task expected fallback** — `(1 − success rate) × cost of handling a failure`. Usually the
largest layer, and the one almost every cost model omits.
**Layer 3 · fixed monthly** — vector store, infrastructure, eval runs, monitoring, maintenance.

Keep them apart. They behave differently as volume grows: layer 1 and 2 are linear and never amortise,
layer 3 gets cheaper per task forever.

In [ ]:
def agent_input_tokens(base, growth, turns):
    """Input tokens an agent sends across a whole run.

    The API is STATELESS. Nothing is remembered between calls, so the whole
    conversation so far is re-sent as input on every turn. Turn t sends
    base + growth*(t-1), and summed over T turns:

        base*T  +  growth * T*(T-1)/2        <- quadratic in turns, not linear

    Keep the two terms in proportion. base*T is LINEAR (the fixed prompt re-sent
    every turn); only the second term is quadratic. The quadratic one overtakes
    the linear one at turns > 2*base/growth + 1 -- about 15 turns for 5,500/800.
    At 6 turns it is only ~27% of the bill, so on short loops shrink `base` first.

    `growth` is everything one completed turn adds to the conversation:

        growth = the model's own reply (its thought + chosen action)
               + the observation the tool returned

    Both halves are re-sent as INPUT on every later turn. So the model's reply is
    paid for twice over: ONCE at the OUTPUT price, on the turn it is written, and
    then at the INPUT price on every turn after that. There is nothing further to
    add for replies -- they are already inside `growth`.
    """
    return base * turns + growth * turns * (turns - 1) // 2


def variable_cost(tier, fresh_in=0, cached_in=0, out=0, retrieval_usd=0.0, tool_usd=0.0):
    "LAYER 1 — per-task variable cost, US dollars."
    p = PRICES[tier]
    assert fresh_in >= 0 and cached_in >= 0 and out >= 0, "token counts cannot be negative"
    return (fresh_in / 1e6 * p["in"]
            + cached_in / 1e6 * p["cached_in"]
            + out / 1e6 * p["out"]
            + retrieval_usd + tool_usd)


def cost_per_successful_task(var_usd, success_rate, failure_usd):
    "LAYER 1 + LAYER 2. The number to quote."
    assert 0.0 <= success_rate <= 1.0, "success rate is a probability, not a percentage"
    assert failure_usd >= 0, "a failure cannot cost less than nothing"
    return var_usd + (1.0 - success_rate) * failure_usd


def monthly(var_usd, success_rate, failure_usd, volume, fixed_monthly_usd=0.0):
    "Everything, for a month."
    return cost_per_successful_task(var_usd, success_rate, failure_usd) * volume + fixed_monthly_usd


def break_even_success_rate(cheap_var, dear_total, failure_usd):
    """The success rate the CHEAP option needs to match the expensive one.

    cheap_var + (1-p) * failure = dear_total   ->   p = 1 - (dear_total - cheap_var)/failure
    """
    p = 1.0 - (dear_total - cheap_var) / failure_usd
    return max(0.0, min(1.0, p))

## Section 3 · Pre-read 3's worked example, reproduced

A support-triage assistant, **50,000 tickets a month**. Four architectures, token cost only for now.

Check these against the brief — they are the same numbers on purpose.

In [ ]:
VOLUME = 50_000

v1 = variable_cost("cheap",    fresh_in=1_500,  out=400)
v2 = variable_cost("cheap",    fresh_in=5_500,  out=400)          # +4,000 tokens of retrieved context

# v3/v4: a six-turn agent. Each completed turn adds ~800 tokens to the conversation:
# ~400 of the model's own reply (thought + action) and ~400 of tool observation.
TURNS, BASE          = 6, 5_500
REPLY, OBSERVATION   = 400, 400
GROWTH               = REPLY + OBSERVATION          # = 800
agent_in  = agent_input_tokens(BASE, GROWTH, TURNS)
agent_out = REPLY * TURNS

v3 = variable_cost("cheap",    fresh_in=agent_in, out=agent_out)
v4 = variable_cost("frontier", fresh_in=agent_in, out=agent_out)

rows = [("v1 · one call", v1), ("v2 · + retrieval", v2),
        ("v3 · 6-turn agent, cheap", v3), ("v4 · same agent, frontier", v4)]
print(f"{'architecture':<30}{'per task':>12}{'per month':>14}")
for name, c in rows:
    print(f"{name:<30}{c:>12.5f}{c*VOLUME:>14,.2f}")

print(f"\nagent input: {BASE:,}x{TURNS} base  +  {GROWTH}x{TURNS*(TURNS-1)//2} carried forward"
      f"  =  {agent_in:,} tokens")
print(f"   of which the model's own replies re-sent as input: "
      f"{REPLY*TURNS*(TURNS-1)//2:,} tokens")
print(f"   the same replies also cost {agent_out:,} tokens at the OUTPUT price, once each.")
print(f"\nv4 is {v4/v3:.0f}x v3 and {round(v4/v1, -1):.0f}x v1, for one feature.")

assert round(v1, 5) == 0.00031 and round(v2, 5) == 0.00071
assert round(v3, 5) == 0.00546 and round(v4, 3) == 0.285
print("\n✓ matches Pre-read 3")

## Section 4 · Add the success rate — and watch the ranking invert

`v3` resolves **55%** of tickets unaided, `v4` resolves **80%**. Every unresolved ticket falls to a human:
eight minutes at a loaded **US$45/hour = $6.00**.

On cost per *token*, `v4` is 52× worse. Now ask the question that matters.

In [ ]:
FAILURE = escalation_cost("support_agent")          # US$6.00

t3 = cost_per_successful_task(v3, 0.55, FAILURE)
t4 = cost_per_successful_task(v4, 0.80, FAILURE)

print(f"v3  tokens {v3:.5f}  + fallback {0.45*FAILURE:.2f}  =  ${t3:.3f} per RESOLVED ticket")
print(f"v4  tokens {v4:.5f}  + fallback {0.20*FAILURE:.2f}  =  ${t4:.3f} per RESOLVED ticket   (the slide rounds to $1.49)")
print(f"\nthe 52x-more-expensive architecture is {1-t4/t3:.0%} CHEAPER per resolved ticket")
print(f"on {VOLUME:,} tickets/month that is ${(t3-t4)*VOLUME:,.0f} a month, "
      f"${(t3-t4)*VOLUME*12:,.0f} a year")

# where does the money actually go?
for label, tok, tot in (("v3", v3, t3), ("v4", v4, t4)):
    print(f"\n{label}: the human fallback is {(tot-tok)/tot:.1%} of cost per resolved ticket")

p = break_even_success_rate(v3, t4, FAILURE)
print(f"\nBREAK-EVEN: v3 wins once its resolution rate reaches {p:.1%}")
assert 0.75 <= p <= 0.76
print("✓ matches Pre-read 3 (75.3%)")

## Section 5 · Never ship a point estimate

The success rate is an estimate. So is the other one. Print the table, not the number.

In [ ]:
def sensitivity(var_usd, failure_usd, centre, spread=0.10, step=0.05, label=""):
    print(f"{label}  cost per successful task, success rate ±{spread:.0%}")
    r = centre - spread
    while r <= centre + spread + 1e-9:
        mark = "  <- estimate" if abs(r - centre) < 1e-9 else ""
        print(f"   {r:6.0%}   ${cost_per_successful_task(var_usd, r, failure_usd):8.3f}{mark}")
        r += step

sensitivity(v3, FAILURE, 0.55, label="v3 ·")
print()
sensitivity(v4, FAILURE, 0.80, label="v4 ·")

print("\nRead it this way: v4 stays cheaper across the whole plausible range,")
print("so the decision is robust. That is worth saying out loud to a sponsor —")
print("a robust answer and a knife-edge answer deserve different amounts of confidence.")

## Section 6 - A2 Problem A measured baseline

Layer 1 uses D5(b) provider usage at the recorded list price. Layer 2 prices a failed first response as 12 minutes of assessor labour. Layer 3 is USD 0 measured provider fees for the local prototype; an illustrative USD 152 maintenance assumption is shown separately. The source aggregates and all five models are in `battery_inputs.json`.


In [ ]:
from cost_model.calculate_d6 import calculate
D6 = calculate()
M = next(r for r in D6['models'] if r['model_id'].startswith('mistralai/') and r['prompt_version'] == 'v2-final')
G = next(r for r in D6['models'] if r['model_id'] == 'openai/gpt-4.1-mini')
MY = dict(volume=8000, tier='mistral_a2', fresh_in=M['prompt_tokens']/55, cached_in=0, out=M['completion_tokens']/55, retrieval_usd=0.0, success_rate=M['passed']/55, failure_usd=escalation_cost('claims_assessor'), fixed_monthly_usd=0.0)
var = variable_cost(MY['tier'], fresh_in=MY['fresh_in'], cached_in=MY['cached_in'], out=MY['out'], retrieval_usd=MY['retrieval_usd'])
per = cost_per_successful_task(var, MY['success_rate'], MY['failure_usd'])
total = monthly(var, MY['success_rate'], MY['failure_usd'], MY['volume'], MY['fixed_monthly_usd'])
assert abs(var-M['layer_1_per_task_usd']) < 1e-6
assert abs(total-M['monthly_usd']) < 0.01
print(f'Mistral V2: {M["passed"]}/55 passed; layer 1 USD {var:.6f}/task; 'f'layer 2 USD {(per-var):.6f}/task')
print(f'Baseline cost USD {per:.6f}/task; USD {total:,.2f}/month')
print(f'Illustrative +4 hours/month review: USD {total+152:,.2f}/month')
sensitivity(var, MY['failure_usd'], MY['success_rate'], label='Mistral V2:')
print('\nFive V2 models, same 55-trial selection:')
for r in D6['models']:
    if r['prompt_version'] == 'v2-final':
        print(f'{r["model_id"]:<45} {r["passed"]}/55  'f'USD {r["all_in_per_task_usd"]:.6f}/task')
need = break_even_success_rate(var, G['all_in_per_task_usd'], MY['failure_usd'])
print(f'Cheap Mistral break-even vs GPT-4.1 Mini: {need:.2%}; 'f'observed {MY["success_rate"]:.2%}')


The full D6 ledger, operational caps, raw-source audit and report section are in `D6_cost_model.md` and the committed `cost_model/` evidence files. This notebook makes no live request.
